# 04a Report: write the CRAS room verdicts into an IFC copy and a BCF file

**Slice.** The 35 room elements of the CRAS lab ("room B") with the verdicts from notebook 03a, written back into a copy of `craslabbim.ifc` (IFC2X3) and into a BCF 2.1 file.

**Question.** Can the verdicts be handed to the people who own the model: as properties on the right elements, and as one issue for every element that is not OK?

Sections: (1) read the verdicts, (2) write them into the IFC copy, (3) write the BCF issues, (4) read both files back, then the answer.
Terms (element, verdict, scan class, oracle labels, tolerance) are defined in [CONTEXT.md](../CONTEXT.md).
Nothing is written outside `outputs/04a/` (gitignored), and the output is the same on every run.

In [1]:
import csv, uuid
import xml.etree.ElementTree as ET
import zipfile
from collections import Counter
from pathlib import Path

import ifcopenshell
import ifcopenshell.api
import ifcopenshell.guid
import ifcopenshell.util.element
from bcf.v2.bcfxml import BcfXml

# All parameters in one place.
IN = Path("../outputs/03a")  # written by notebook 03a
SRC_IFC = Path("../data/cras/craslabbim.ifc")
OUT = Path("../outputs/04a")  # gitignored; never write outside outputs/
OUT.mkdir(parents=True, exist_ok=True)
GUID_NAMESPACE = uuid.UUID("5ca17b13-0000-4000-8000-000000000000")  # fixed, so the new entities' GlobalIds are the same on every run
PSET = "ScanVsBim_Verdict"
TOL = 0.015  # USIBD LOA30: at most 15 mm at 2 sigma (95 %); quoted in 03a section 3

## 1. Read the verdicts

The reported verdict is the **no-labels** one: it is what the pipeline would produce on a real scan, where nobody hands over the scan classes. The **oracle-labels** verdict is the upper bound and is written alongside as separate properties, so a reader sees how much the scan classes would change.
Offsets are signed, positive into the room, except for a wall with two room-facing sides (a partition standing inside the room box): its offsets are positive along its reference side's normal (03a's wall side test; CONTEXT.md: reference side).

In [2]:
def read(variant):
    with open(IN / f"verdicts_{variant}.csv", newline="") as f:
        return {r["global_id"]: r for r in csv.DictReader(f)}


rows, oracle = read("no_labels"), read("oracle_labels")
assert len(rows) == 35 and set(rows) == set(oracle)
not_ok = {g: r for g, r in rows.items() if r["verdict"] != "OK"}
print("no labels:    ", dict(Counter(r["verdict"] for r in rows.values())))
print("oracle labels:", dict(Counter(r["verdict"] for r in oracle.values())))

no labels:     {'not confirmed': 29, 'shifted': 1, 'OK': 5}
oracle labels: {'not confirmed': 27, 'OK': 7, 'shifted': 1}


## 2. Write the verdicts into a copy of the IFC

One property set `ScanVsBim_Verdict` on each of the 35 room elements: `Verdict`, `PointCount`, `MedianOffset_mm` and, for elements that are not confirmed, `Reason` (all no labels), plus `Verdict_OracleLabels` and `MedianOffset_OracleLabels_mm`. Other elements get nothing. The geometry is not touched, so the file is never tessellated.
The new property sets and their relationships get GlobalIds derived from the element's GlobalId; the model's own GlobalIds are kept.

In [3]:
f = ifcopenshell.open(str(SRC_IFC))
assert f.schema == "IFC2X3"
for gid, r in rows.items():
    el = f.by_guid(gid)
    assert el.is_a() == r["ifc_class"], (gid, el.is_a(), r["ifc_class"])
    props = {"Verdict": r["verdict"], "PointCount": int(r["point_count"]), "Verdict_OracleLabels": oracle[gid]["verdict"]}
    for key, row in (("MedianOffset_mm", r), ("MedianOffset_OracleLabels_mm", oracle[gid])):
        if row["median_signed_offset_mm"]:  # blank when an element has no points: leave the property out
            props[key] = round(float(row["median_signed_offset_mm"]), 1)
    if r["reason"]:
        props["Reason"] = r["reason"]
    pset = ifcopenshell.api.run("pset.add_pset", f, product=el, name=PSET)
    ifcopenshell.api.run("pset.edit_pset", f, pset=pset, properties=props)
    for entity in [pset, *f.get_inverse(pset)]:  # the pset and its IfcRelDefinesByProperties got random GlobalIds
        if entity.is_a("IfcRoot"):
            entity.GlobalId = ifcopenshell.guid.compress(uuid.uuid5(GUID_NAMESPACE, f"{entity.is_a()}:{gid}").hex)
REPORT_IFC = OUT / "craslabbim_verdicts.ifc"
f.write(str(REPORT_IFC))
print(f"{REPORT_IFC.name}: {len(rows)} elements carry {PSET}")

craslabbim_verdicts.ifc: 35 elements carry ScanVsBim_Verdict


## 3. One BCF issue per element that is not OK

Each issue has a viewpoint that selects the element by GlobalId. The title and description give the verdict, the offset, why an element is not confirmed, and what the oracle labels would have said.

In [4]:
BCF_PATH = OUT / "craslabbim_verdicts.bcf"
bcf = BcfXml.create_new("CRAS room B")
for gid, r in not_ok.items():
    offset = f"{float(r['median_signed_offset_mm']):+.1f} mm" if r["median_signed_offset_mm"] else "none"
    if r["verdict"] == "not confirmed":  # the median of too few or inconsistent points is not a measurement of the element
        found = f"Not confirmed because: {r['reason']} (median of those points {offset}, not a measured offset)."
    else:
        found = f"Median offset {offset} (positive into the room)."
    wall = " Wall offset: for a wall with two room-facing sides, positive along its reference side's normal; a wall whose side the scan cannot tell can still fold at half its thickness (03a, issue #17)." if r["ifc_class"] == "IfcWallStandardCase" else ""
    topic = bcf.add_topic(
        title=f"{r['name']} ({r['ifc_class']}): {r['verdict']}",
        description=f"{r['point_count']} scan points. {found} Tolerance {TOL * 1000:.0f} mm, USIBD LOA30.{wall} "
                    f"With oracle labels: {oracle[gid]['verdict']}.",
        author="scan-vs-bim", topic_type="Issue", topic_status="Open")
    topic.add_viewpoint(f.by_guid(gid))
bcf.save(BCF_PATH)
print(f"{BCF_PATH.name}: {len(not_ok)} issues")

craslabbim_verdicts.bcf: 30 issues


## 4. Read both files back

Re-open the IFC and check every element's properties; open the BCF zip and check its version, its topic count and that the selected GlobalIds are exactly the not-OK elements.

In [5]:
back = ifcopenshell.open(str(REPORT_IFC))
with_pset = set()
for el in back.by_type("IfcElement"):
    props = ifcopenshell.util.element.get_psets(el).get(PSET)
    if props is not None:
        with_pset.add(el.GlobalId)
        assert props["Verdict"] == rows[el.GlobalId]["verdict"] and props["Verdict_OracleLabels"] == oracle[el.GlobalId]["verdict"]
        if rows[el.GlobalId]["median_signed_offset_mm"]:
            assert abs(props["MedianOffset_mm"] - float(rows[el.GlobalId]["median_signed_offset_mm"])) < 0.06
assert with_pset == set(rows)
guids = [e.GlobalId for e in back.by_type("IfcRoot")]
assert len(guids) == len(set(guids))
print("properties re-read on", len(with_pset), "elements; no other element has them; all GlobalIds unique")

with zipfile.ZipFile(BCF_PATH) as z:
    version = ET.fromstring(z.read("bcf.version")).attrib["VersionId"]
    markups = [n for n in z.namelist() if n.endswith("markup.bcf")]
    selected = [c.attrib["IfcGuid"] for n in z.namelist() if n.endswith(".bcfv") for c in ET.fromstring(z.read(n)).iter("Component")]
assert version == "2.1" and len(markups) == len(not_ok) and set(selected) == set(not_ok)
print(f"BCF {version}: {len(markups)} topics, {len(set(selected))} distinct selected GlobalIds, equal to the not-OK elements")

properties re-read on 35 elements; no other element has them; all GlobalIds unique
BCF 2.1: 30 topics, 30 distinct selected GlobalIds, equal to the not-OK elements



## Answer

Yes. Without oracle labels the 35 room elements get 5 OK, 1 shifted and 29 not confirmed; with oracle labels 7 OK, 1 shifted and 27 not confirmed. The no-labels verdicts are the reported ones.

**IFC.** `outputs/04a/craslabbim_verdicts.ifc` re-opens in IfcOpenShell (IFC2X3). All 35 room elements carry `ScanVsBim_Verdict` with the no-labels verdict, offset in mm (positive into the room; along the reference side's normal for a wall with two room-facing sides), point count and reason, and the oracle-label verdict and offset alongside. No other element has the property set, and every GlobalId in the file is unique. An element with no points under the oracle labels has no oracle offset property.

**BCF.** `outputs/04a/craslabbim_verdicts.bcf` is BCF 2.1 with 30 issues, one for each of the 30 elements that are not OK (29 not confirmed, 1 shifted). Each issue's viewpoint selects that element's GlobalId; the 30 selected GlobalIds are exactly the not-OK elements.

Most issues are therefore "not confirmed", not "shifted": the scan does not show that these elements are wrong, only that it cannot show that they are right.
